# Association Rule Based Recommender System

Online Retail II veri seti üzerinde birliktelik kuralı öğrenimi ile ürün tavsiye sistemi çalışması.

## İş Problemi

Aşağıda 3 farklı kullanıcının sepet bilgileri verilmiştir. Bu sepet bilgilerine en uygun ürün önerisini birliktelik kuralı kullanarak yapınız.
Ürün önerileri 1 tane ya da 1'den fazla olabilir. Karar kurallarını 2010-2011 Germany müşterileri üzerinden türetiniz.

- **Kullanıcı 1** sepet ürün id: `21987`
- **Kullanıcı 2** sepet ürün id: `23235`
- **Kullanıcı 3** sepet ürün id: `22747`

## Veri Seti Hikayesi

**Online Retail II** isimli veri seti İngiltere merkezli bir perakende şirketinin 01/12/2009 - 09/12/2011 tarihleri arasındaki online satış işlemlerini içeriyor.
Şirketin ürün kataloğunda hediyelik eşyalar yer almaktadır ve çoğu müşterisinin toptancı olduğu bilgisi mevcuttur.

**Değişkenler (8 Değişken, 541.909 Gözlem)**

- **InvoiceNo:** Fatura Numarası (C ile başlıyorsa işlem iptal)
- **StockCode:** Ürün kodu (her ürün için eşsiz)
- **Description:** Ürün ismi
- **Quantity:** Ürün adedi
- **InvoiceDate:** Fatura tarihi
- **UnitPrice / Price:** Fatura fiyatı (Sterlin)
- **CustomerID:** Eşsiz müşteri numarası
- **Country:** Ülke ismi

## Görevler

1. **GÖREV 1:** Veriyi Hazırlama
2. **GÖREV 2:** Alman Müşteriler Üzerinden Birliktelik Kuralları Üretme
3. **GÖREV 3:** Sepet İçerisindeki Ürün Id'leri Verilen Kullanıcılara Ürün Önerisinde Bulunma

In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

---
## GÖREV 1: Veriyi Hazırlama

### Adım 1: Online Retail II veri setinden 2010-2011 sheet'ini okutunuz.

In [2]:
df_ = pd.read_excel("datasets/online_retail_II.xlsx", sheet_name="Year 2010-2011")
df = df_.copy()
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Adım 2: StockCode'u POST olan gözlem birimlerini drop ediniz.

In [3]:
# "StockCode" değeri "POST" olan satırları veri setinden çıkartıyoruz.
df = df[df["StockCode"] != "POST"]
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Adım 3: Boş değer içeren gözlem birimlerini drop ediniz.

In [ ]:
# Boş (eksik) değer içeren tüm satırları veri setinden çıkartıyoruz.
df.dropna(inplace=True)
# Boş değerlerin başarıyla silindiğini kontrol ediyoruz.
df.isnull().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

### Adım 4: Invoice içerisinde C bulunan değerleri veri setinden çıkarınız.

In [5]:
df = df[~df["Invoice"].str.contains("C", na=False)]
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Adım 5: Price değeri sıfırdan küçük olan gözlem birimlerini filtreleyiniz.

In [6]:
df = df[df["Quantity"] > 0]
df = df[df["Price"] > 0]
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Quantity,396785.0,13.016349,1.0,2.0,6.0,12.0,80995.0,179.579125
InvoiceDate,396785,2011-07-10 23:58:18.325340,2010-12-01 08:26:00,2011-04-07 11:12:00,2011-07-31 14:39:00,2011-10-20 14:41:00,2011-12-09 12:50:00,NaN
Price,396785.0,3.037677,0.001,1.25,1.95,3.75,4161.06,17.829741
Customer ID,396785.0,15301.463886,12346.0,13975.0,15159.0,16801.0,18287.0,1709.852311


### Adım 6: Price ve Quantity değişkenlerinin aykırı değerlerini inceleyiniz, gerekirse baskılayınız.

In [7]:
def outlier_thresholds(dataframe, variable):
    quartile1 = dataframe[variable].quantile(0.01)
    quartile3 = dataframe[variable].quantile(0.99)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    return low_limit, up_limit


def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit


df["Quantity"] = df["Quantity"].astype(float)
df["Price"] = df["Price"].astype(float)
replace_with_thresholds(df, "Quantity")
replace_with_thresholds(df, "Price")
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Quantity,396785.0,11.855703,1.0,2.0,6.0,12.0,298.5,25.55376
InvoiceDate,396785,2011-07-10 23:58:18.325340,2010-12-01 08:26:00,2011-04-07 11:12:00,2011-07-31 14:39:00,2011-10-20 14:41:00,2011-12-09 12:50:00,NaN
Price,396785.0,2.835605,0.001,1.25,1.95,3.75,31.56,2.982371
Customer ID,396785.0,15301.463886,12346.0,13975.0,15159.0,16801.0,18287.0,1709.852311


---
## GÖREV 2: Alman Müşteriler Üzerinden Birliktelik Kuralları Üretme

### Adım 1: `create_invoice_product_df` fonksiyonunu tanımlayınız.

Fatura ürün pivot table'i oluşturacak fonksiyon. `id=True` olduğunda sütunlarda StockCode kullanılır.

In [ ]:
# Bu fonksiyon, fatura bazında hangi ürünlerin alındığını gösteren bir pivot tablo oluşturur.
# Eğer id=True olarak verilirse, sütunlarda ürünlerin StockCode'u (benzersiz ürün kodu) kullanılır.
# Eğer id=False olarak verilirse, sütunlarda ürün açıklaması (Description) kullanılır.
# Her hücredeki değer, ilgili faturada o üründen en az bir adet alındıysa 1, alınmadıysa 0 olur.

def create_invoice_product_df(dataframe, id=False):
    if id:
        # "Invoice" ve "StockCode" değerlerine göre gruplayıp, her grupta satılan toplam ürün adedini hesaplıyoruz.
        # Daha sonra her StockCode'u sütun olarak yerleştirip eksik değerleri 0 ile dolduruyoruz.
        pivot = dataframe.groupby(["Invoice", "StockCode"])["Quantity"].sum().unstack().fillna(0)
    else:
        # id=False ise, bu sefer ürün açıklamalarını sütunlarda göstermek için aynı işlemi "Description" ile yapıyoruz.
        pivot = dataframe.groupby(["Invoice", "Description"])["Quantity"].sum().unstack().fillna(0)
    # Burada, her hücredeki değer sıfırdan büyükse 1, değilse 0 olacak şekilde binary bir tablo elde ediyoruz.
    return (pivot > 0).astype(int)

# Şimdi, Alman ("Germany") müşterilere ait kayıtları filtreliyoruz.
df_de = df[df["Country"] == "Germany"]

# Ardından, yukarıdaki fonksiyon ile bu müşterilere özel, fatura-ürün bazlı bir tablo oluşturuyoruz (ürün kodu ile).
de_inv_pro_df = create_invoice_product_df(df_de, id=True)

# Son olarak, oluşan tablonun ilk 5 satır ve ilk 5 sütununu ekrana getiriyoruz.
de_inv_pro_df.iloc[:5, :5]

StockCode,10002,10125,10135,11001,15034
Invoice,,,,,
536527,0,0,0,0,0
536840,0,0,0,0,0
536861,0,0,0,0,0
536967,0,0,0,0,0
536983,0,0,0,0,0


### Adım 2: `create_rules` fonksiyonunu tanımlayınız ve Alman müşteriler için kuralları bulunuz.

In [9]:
# create_rules fonksiyonu, belirli bir ülkeye ait (örn. Germany) verilerle birliktelik kuralları (association rules) oluşturur. 
# Fonksiyonun amacı, birlikte satın alınan ürünleri bulmaktır.

def create_rules(dataframe, id=True, country="Germany"):
    # İlk olarak, ilgili ülkeye ait müşteri verilerini filtreleriz.
    dataframe = dataframe[dataframe["Country"] == country]
    
    # create_invoice_product_df fonksiyonu ile, fatura bazında ürünlerin 1/0 (alındı/alınmadı) formatında tablosunu oluştururuz.
    dataframe = create_invoice_product_df(dataframe, id)
    
    # apriori algoritmasını kullanarak, birlikte en az %1 oranında bulunan ürün kümelerini (frequent itemsets) buluruz.
    frequent_itemsets = apriori(dataframe.astype(bool), min_support=0.01, use_colnames=True)
    
    # association_rules fonksiyonu ile, yukarıdaki itemsetlerden birliktelik kuralları elde edilir.
    # Burada 'support' metriği kullanılır ve en az %1 destek eşiği (min_threshold) belirtilir.
    rules = association_rules(frequent_itemsets, metric="support", min_threshold=0.01)
    return rules

# Fonksiyonu çağırıp, Alman müşterileri için oluşturulan birliktelik kurallarının ilk satırlarını görüntülüyoruz.
rules = create_rules(df, id=True, country="Germany")
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({16237}),frozenset({22326}),0.011136,0.249443,0.011136,1.000000,4.008929,1.0,0.008358,inf,0.759009,0.044643,1.000000,0.522321
1,frozenset({22326}),frozenset({16237}),0.249443,0.011136,0.011136,0.044643,4.008929,1.0,0.008358,1.035073,1.000000,0.044643,0.033884,0.522321
2,frozenset({20674}),frozenset({20675}),0.022272,0.033408,0.013363,0.600000,17.960000,1.0,0.012619,2.416481,0.965831,0.315789,0.586175,0.500000
3,frozenset({20675}),frozenset({20674}),0.033408,0.022272,0.013363,0.400000,17.960000,1.0,0.012619,1.629547,0.976959,0.315789,0.386333,0.500000
4,frozenset({20674}),frozenset({20676}),0.022272,0.037862,0.011136,0.500000,13.205882,1.0,0.010293,1.924276,0.945330,0.227273,0.480324,0.397059


---
## GÖREV 3: Sepet İçerisindeki Ürün Id'leri Verilen Kullanıcılara Ürün Önerisinde Bulunma

### Adım 1: `check_id` fonksiyonunu kullanarak verilen ürünlerin isimlerini bulunuz.

In [10]:
# check_id fonksiyonu, verilen bir ürünün StockCode'una karşılık gelen ürün ismini ("Description") bulmak için kullanılmaktadır.
# Fonksiyonun içine bir veri çerçevesi (dataframe) ve aranan ürün kodu (stock_code) girilir.
# Fonksiyon, StockCode'u verilen ürünün Description sütunundaki değerini bulur ve yazdırır.

def check_id(dataframe, stock_code):
    product_name = dataframe[dataframe["StockCode"] == stock_code][["Description"]].values[0].tolist()
    print(product_name)

# Aşağıda ise, belirli üç ürün kodu için, ürüne karşılık gelen isimlerin ekrana yazdırılması amaçlanmaktadır.
for product_id in [21987, 23235, 22747]:
    print(f"Ürün ID: {product_id}")
    check_id(df_de, product_id)

Ürün ID: 21987
['PACK OF 6 SKULL PAPER CUPS']
Ürün ID: 23235
['STORAGE TIN VINTAGE LEAF']
Ürün ID: 22747
["POPPY'S PLAYHOUSE BATHROOM"]


### Adım 2: `arl_recommender` fonksiyonunu kullanarak 3 kullanıcı için ürün önerisinde bulununuz.

In [11]:
# arl_recommender fonksiyonu, birliktelik kuralları (rules_df) içinden verilen bir ürün ID'si (product_id) için 
# önerilebilecek ürünleri bulmak amacıyla kullanılır.
# Bu fonksiyon, önce kuralları "lift" değerine göre büyükten küçüğe sıralar.
# Ardından, her satırdaki "antecedents" (öncüller) kümesinde, öneri yapılacak ürün ID'si var mı diye bakar.
# Eğer varsa, aynı satırdaki "consequents" (sonuç kümeleri) içindeki ürünü öneri listesine ekler.
# Son olarak, istenilen sayıda öneriyi (rec_count kadar) döndürür.

def arl_recommender(rules_df, product_id, rec_count=1):
    # Kuralları 'lift' değerine göre azalan şekilde sıralıyoruz
    sorted_rules = rules_df.sort_values("lift", ascending=False)
    recommendation_list = []
    # Her kuraldaki antecedent'larda (öncüller) product_id olup olmadığına bak
    for i, product in enumerate(sorted_rules["antecedents"]):
        for j in list(product):
            if j == product_id:
                # Eğer product_id varsa, o kuraldaki consequents (sonuç) ürün kodunu önerilere ekle
                recommendation_list.append(list(sorted_rules.iloc[i]["consequents"])[0])
    # İstenilen sayıda öneriyi döndür
    return recommendation_list[0:rec_count]

# Fonksiyonu çağırınca, 21987 ürün kodu için 1 tane ürün önerisi görürüz
arl_recommender(rules, 21987, 1)

[21988]

In [12]:
arl_recommender(rules, 23235, 1)

[23244]

In [13]:
arl_recommender(rules, 22747, 1)

[22746]

### Adım 3: Önerilecek ürünlerin isimlerine bakınız.

In [14]:
# Burada, her bir kullanıcı/basket_id için arl_recommender fonksiyonu kullanılarak önerilen ürünlerin olduğu bir sözlük oluşturuluyor.
# Her kullanıcı için öncelikle sepetinde bulunan ürünün ismi ekrana yazdırılıyor ('check_id' fonksiyonu ile ürün ismi bulunur).
# Ardından, ARL ile önerilen ürünün StockCode'u alınarak, bu ürünün ismi de ekrana bastırılıyor.
# "print('-' * 50)" satırı hem okuma kolaylığı hem de kullanıcılar arası ayrım yapmak için kullanılmıştır.

recommendations = {
    21987: arl_recommender(rules, 21987, 1)[0],
    23235: arl_recommender(rules, 23235, 1)[0],
    22747: arl_recommender(rules, 22747, 1)[0],
}

for basket_id, rec_id in recommendations.items():
    print(f"Sepetteki ürün ({basket_id}):", end=" ")
    check_id(df_de, basket_id)
    print(f"Önerilen ürün ({rec_id}):", end=" ")
    check_id(df_de, rec_id)
    print("-" * 50)

Sepetteki ürün (21987): ['PACK OF 6 SKULL PAPER CUPS']
Önerilen ürün (21988): ['PACK OF 6 SKULL PAPER PLATES']
--------------------------------------------------
Sepetteki ürün (23235): ['STORAGE TIN VINTAGE LEAF']
Önerilen ürün (23244): ['ROUND STORAGE TIN VINTAGE LEAF']
--------------------------------------------------
Sepetteki ürün (22747): ["POPPY'S PLAYHOUSE BATHROOM"]
Önerilen ürün (22746): ["POPPY'S PLAYHOUSE LIVINGROOM "]
--------------------------------------------------
